In [60]:
%cd /mlx_devbox/users/janne.spijkervet/repo/333/samantha/
%load_ext autoreload
%autoreload 2

/mlx_devbox/users/janne.spijkervet/repo/333/samantha
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import os
from samantha.utils.hdfs_helper import hdfs_ls, get

# arnold_task_id = 558104
# hdfs_ckpt_dir = f"hdfs://harunava/home/byte_arnold_va/data/lab/audio/soundstorm/tasks/{arnold_task_id}/trials"
# checkpoints = list(
#     filter(lambda f: ".ckpt" in f and "step=" in f, hdfs_ls(f"-R {hdfs_ckpt_dir}"))
# )
# checkpoints.reverse()

# get(checkpoints[0], "model.ckpt")

# get("hdfs://harunava/home/byte_arnold_va/data/lab/audio/soundstorm/tasks/558104/trials/2808839/output/soundstorm/baseline/checkpoints/epoch=0-step=21500.ckpt", "558104.ckpt")

In [46]:
ckpt = torch.load("epoch=0-step=105000.ckpt")

In [61]:
from recipes.soundstorm.lightning.soundstorm import SoundStorm
from recipes.soundstorm.inference.semantic2audio import load_model

device = "cuda"

# semantic_ckpt = "/mnt/bn/audio-diffusion/ckpts/musiclm/semantic_flash_llama/mcc40m/step=040000-tr_loss=2.3858.ckpt"
soundstorm = load_model("epoch=0-step=105000.ckpt", device, semantic_ckpt=semantic_ckpt)

/usr/local/lib/python3.9/dist-packages/pytorch_lightning/utilities/parsing.py:197: UserWarning: Attribute 'masking_scheme' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['masking_scheme'])`.
  rank_zero_warn(


In [62]:
# %cd /mlx_devbox/users/janne.spijkervet/repo/333/samantha/
# %load_ext autoreload
# %autoreload 2
# from recipes.soundstorm.inference.text2audio import generate

# iterations = [32,32,32,32,1,1,1,1,1,1,1,1]
# score_strategies = ["random", "random", "random", "random", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit"]


# texts = ["jazz music"]
# sampled_t = None
# temperature = 1.0

# audio = generate(soundstorm, texts, 1, max_seq_len=500, iterations=iterations, score_strategies=score_strategies, sampled_t=sampled_t, temperature=temperature)


In [63]:
# import IPython.display as ipd

# ipd.display(ipd.Audio(audio[0].cpu(), rate=24000))

In [64]:
import torch
import pandas as pd

semantic_tokens = torch.load("/mnt/bn/audio-diffusion/assets/semantic_decoder_gen_samples_100x250.pt")
semantic_tokens = semantic_tokens[:, None].to(device)


df = pd.read_csv("/mnt/bn/audio-diffusion/data/google_prompts/text_prompt_collection_20230523.csv")
text_prompts = df["text"].tolist()[:len(semantic_tokens)]

In [65]:
from recipes.soundstorm.inference.semantic2audio import generate as generate_from_semantic

audio = generate_from_semantic(soundstorm, semantic_tokens[0], 500, iterations, score_strategies)
ipd.display(ipd.Audio(audio[0].cpu(), rate=24000))

Iteratively decoding audio tokens...: 100%|█████| 12/12 [00:07<00:00,  1.69it/s]


In [70]:
# hi


from recipes.soundstorm.lightning.soundstorm import SoundStormInference

soundstorm_ckpt = "epoch=0-step=105000.ckpt"
semantic_ckpt = "/mnt/bn/audio-diffusion/ckpts/musiclm/semantic_flash_llama/mcc40m/step=040000-tr_loss=2.3858.ckpt"
soundstorm_inference = SoundStormInference(soundstorm_ckpt, semantic_ckpt)



/usr/local/lib/python3.9/dist-packages/pytorch_lightning/utilities/parsing.py:197: UserWarning: Attribute 'masking_scheme' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['masking_scheme'])`.
  rank_zero_warn(
Some weights of the model checkpoint at bert-large-uncased were not used when initializing BertModel: ['cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT ex

In [89]:
soundstorm_inference.semantic_model.requires

{'ssl_frontend': SSLFrontend(
   (stft): Stft(n_fft=1024, win_length=600, hop_length=240, center=True, normalized=False, onesided=True)
   (logmel): LogMel(sr=24000, n_fft=1024, n_mels=80, fmin=0, fmax=12000.0, htk=False)
 ),
 'semantic': RecursiveScriptModule(
   original_name=SemanticModule
   (frontend): RecursiveScriptModule(
     original_name=DefaultFrontend
     (stft): RecursiveScriptModule(original_name=Stft)
     (logmel): RecursiveScriptModule(original_name=LogMel)
   )
   (feature_encoder): RecursiveScriptModule(
     original_name=Conv2dSubsampling
     (conv): RecursiveScriptModule(
       original_name=Sequential
       (0): RecursiveScriptModule(original_name=Conv2d)
       (1): RecursiveScriptModule(original_name=ReLU)
       (2): RecursiveScriptModule(original_name=Conv2d)
       (3): RecursiveScriptModule(original_name=ReLU)
     )
     (out): RecursiveScriptModule(
       original_name=Sequential
       (0): RecursiveScriptModule(original_name=LayerNorm)
       (1):

In [78]:
soundstorm_inference = soundstorm_inference.to("cuda")

In [87]:
text = ["gabber dance track with pumping kick drums"]
audio = soundstorm_inference.generate(text, 1, 500, iterations, score_strategies)

Iteratively decoding audio tokens...: 100%|█████| 12/12 [00:07<00:00,  1.71it/s]


In [88]:
ipd.display(ipd.Audio(audio[0].cpu(), rate=24000))

In [51]:
semantic_ckpt_statedict =  torch.load(semantic_ckpt)

In [57]:
semantic_module = semantic_module.load_state_dict(semantic_ckpt_statedict["state_dict"])

In [58]:
text = ["jazz music"]
mulan_tokens = soundstorm.sample_mulan_tokens(text)
semantic_tokens2 = soundstorm.sample_semantic_tokens(mulan_tokens)

print(semantic_tokens[0])
print(semantic_tokens2)

audio = generate_from_semantic(soundstorm, semantic_tokens2, 500, iterations ,score_strategies)
ipd.display(ipd.Audio(audio[0].cpu(), rate=24000))

Semantic [0 - 250]: 100%|█████████████████████| 250/250 [00:05<00:00, 45.71it/s]


tensor([[ 712,  238,  272,  307,  307,   79,  930,  930,  826,  826,  492,   48,
          172,  856,   57,  209,  289,  116,   78,  874,  960,  172,  417,  913,
          868,  137,   81,  874,   67,   67,   67,   67,   67,  271,  987,  499,
          573,  573,  891,  671,  671,   99,  868,  137,  638,  414,  414,  741,
          341,  627,   50,  465,  433,  362,  883,  418,  991,  204,  692,  692,
          450,  831,   81,  671,  856,   17,   54,  581,  833,  347,  907,  617,
          874,   79,   79,   67,  826,  197,  743,  878, 1016,  666,  666,  666,
          666,  666,  548,  988,   72,  874, 1009, 1009,  874,  138,  913,  911,
           24,   81,  352,  352,  271,  241,   78,  378,  590,  772,   81,  672,
          666,  829,  826,  140,  799,  347,  807,  328,  812,  812,  500,  500,
          500,  951,  590,  831,   81,  352,  352,  352,  855,  871,  613,  548,
          592,  362,  952,  402,  237,  191,   54,  282,  420,  112,   81,  874,
          138,  402,  402,  

Iteratively decoding audio tokens...: 100%|█████| 12/12 [00:07<00:00,  1.71it/s]


In [47]:
import re
import unicodedata

def slugify(value, allow_unicode=False):
    """
    Taken from https://github.com/django/django/blob/master/django/utils/text.py
    Convert to ASCII if 'allow_unicode' is False. Convert spaces or repeated
    dashes to single dashes. Remove characters that aren't alphanumerics,
    underscores, or hyphens. Convert to lowercase. Also strip leading and
    trailing whitespace, dashes, and underscores.
    """
    value = str(value)
    if allow_unicode:
        value = unicodedata.normalize("NFKC", value)
    else:
        value = (
            unicodedata.normalize("NFKD", value)
            .encode("ascii", "ignore")
            .decode("ascii")
        )
    value = re.sub(r"[^\w\s-]", "", value.lower())
    return re.sub(r"[-\s]+", "-", value).strip("-_")



In [48]:
import torchaudio
import os
from recipes.soundstorm.inference.semantic2audio import generate

# iterations = [32,32,32,32,8,8,8,8,8,8,8,8]
# score_strategies = ["random", "random", "random", "random", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit"]

iterations = [32,32,32,32,1,1,1,1,1,1,1,1]
score_strategies = ["random", "random", "random", "random", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit"]

out_dir = "generated_2808839"
os.makedirs(out_dir, exist_ok=True)
            
for st, text in zip(semantic_tokens, text_prompts):
    audio = generate(soundstorm, st, max_seq_len=500, iterations=iterations, score_strategies=score_strategies)    
    torchaudio.save(os.path.join(out_dir, f"{slugify(text)}.wav"), audio[0].cpu(), 24000)

Iteratively decoding audio tokens...: 100%|█████| 12/12 [00:09<00:00,  1.20it/s]
